In [1]:
import os
import certifi
import random
import warnings
from pathlib import Path

from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from tqdm import tqdm

warnings.filterwarnings("ignore")
os.environ["SSL_CERT_FILE"] = certifi.where()

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# Пути к данным
IMAGES_DIR = Path('D:\\Other\\Other_ML\\ITMO CV\\lab1\\Confirmed_fronts').resolve()

IMG_SIZE = 128
BATCH_SIZE = 64
NUM_EPOCHS_CUSTOM = 20        # Эпохи для самописной модели
NUM_EPOCHS_PRETRAINED = 8     # Эпохи для дообучения
LR_CUSTOM = 3e-3
LR_PRETRAINED = 1e-3
NUM_WORKERS = 0               # 0 — без мультипроцессинга
MAX_SAMPLES_PER_CLASS = 10000 # Ограничение выборки для скорости
MIN_SAMPLES_PER_CLASS = 500   # Минимум изображений на класс
EARLY_STOP_PATIENCE = 4       # Остановка, если F1 не растёт N эпох
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Используем устройство: {DEVICE}")

  Используем устройство: cuda


In [ ]:
def build_data(images_dir: Path):
    records = []

    for img_path in images_dir.rglob("*.jpg"):
        parts = img_path.stem.split("$$")
        if len(parts) < 7:
            continue

        color = parts[-4].strip()
        adv_key = f"{parts[-3]}$${parts[-2]}"

        records.append((str(img_path), color, adv_key))

    records = [(p, c, k) for p, c, k in records if c and c != "Unlisted"]

    records.sort(key=lambda r: r[0])
    seen_keys: set = set()
    unique_records: list = []
    for path, color, key in records:
        if key not in seen_keys:
            seen_keys.add(key)
            unique_records.append((path, color))

    # Фильтруем редкие классы
    color_counts = Counter(c for _, c in unique_records)
    valid_colors = {c for c, n in color_counts.items() if n >= MIN_SAMPLES_PER_CLASS}
    unique_records = [(p, c) for p, c in unique_records if c in valid_colors]

    # Ограничиваем MAX_SAMPLES_PER_CLASS на класс
    rng = random.Random(SEED)
    by_color: dict[str, list] = {}
    for p, c in unique_records:
        by_color.setdefault(c, []).append(p)

    sampled: list = []
    for color, img_paths in by_color.items():
        if len(img_paths) > MAX_SAMPLES_PER_CLASS:
            img_paths = rng.sample(img_paths, MAX_SAMPLES_PER_CLASS)
        for p in img_paths:
            sampled.append((p, color))

    classes = sorted({c for _, c in sampled})
    class_to_idx = {c: i for i, c in enumerate(classes)}
    paths = np.array([p for p, _ in sampled])
    labels = np.array([class_to_idx[c] for _, c in sampled])

    print(f"  Всего изображений (после субсэмплинга): {len(paths)}")
    print(f"  Классы ({len(classes)}): {classes}")
    print(f"\n  Распределение по цветам:")
    for color in classes:
        cnt = int((labels == class_to_idx[color]).sum())
        print(f"  {color} {cnt}")
    print()

    return paths, labels, classes, class_to_idx

In [5]:
build_data(IMAGES_DIR)
None

  Всего изображений (после субсэмплинга): 50797
  Классы (11): ['Beige', 'Black', 'Blue', 'Brown', 'Green', 'Grey', 'Orange', 'Red', 'Silver', 'White', 'Yellow']

  Распределение по цветам:
  Beige 557
  Black 10000
  Blue 7717
  Brown 836
  Green 709
  Grey 8618
  Orange 518
  Red 5605
  Silver 7119
  White 8497
  Yellow 621



In [6]:
def get_transforms(train: bool = True):
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
        ])

class MyDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[idx])


def create_dataloaders(paths, labels, num_classes):
    # 64% train, 16% val, 20% test
    tr_paths, te_paths, tr_labels, te_labels = train_test_split(
        paths, labels, test_size=0.2, random_state=SEED, stratify=labels,
    )
    tr_paths, va_paths, tr_labels, va_labels = train_test_split(
        tr_paths, tr_labels, test_size=0.2, random_state=SEED, stratify=tr_labels,
    )

    train_ds = MyDataset(tr_paths, tr_labels, transform=get_transforms(train=True))
    val_ds = MyDataset(va_paths, va_labels, transform=get_transforms(train=False))
    test_ds  = MyDataset(te_paths, te_labels, transform=get_transforms(train=False))

    # Взвешенный сэмплер для борьбы с дисбалансом классов
    class_counts = np.bincount(tr_labels, minlength=num_classes).astype(float)
    weights = 1.0 / (class_counts + 1e-6)
    sample_weights = weights[tr_labels]
    sampler = WeightedRandomSampler(
        sample_weights, num_samples=len(sample_weights), replacement=True,
    )

    pin = DEVICE.type == "cuda"

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=pin,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=pin,
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=pin,
    )

    print(f"  Train: {len(train_ds)},  Val: {len(val_ds)},  Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader

In [7]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int,
                 stride: int = 1, downsample: nn.Module = None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.downsample = downsample

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return F.relu(out, inplace=True)


class CustomResNet(nn.Module):
    """
    Каналы 32 → 64 → 128 → 256  (легче ResNet18)
    """

    def __init__(self, num_classes, channels=(32, 64, 128, 256), blocks_per_stage=(2, 2, 2, 2)):
        super().__init__()
        self.in_channels = channels[0]

        self.conv1 = nn.Conv2d(3, channels[0], kernel_size=3, stride=2,
                               padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels[0])
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(channels[0], blocks_per_stage[0], stride=1)
        self.layer2 = self._make_layer(channels[1], blocks_per_stage[1], stride=2)
        self.layer3 = self._make_layer(channels[2], blocks_per_stage[2], stride=2)
        self.layer4 = self._make_layer(channels[3], blocks_per_stage[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(channels[3], num_classes)

        self._init_weights()

    def _make_layer(self, out_ch: int, blocks: int, stride: int):
        downsample = None
        if stride != 1 or self.in_channels != out_ch:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        layers = [ResidualBlock(self.in_channels, out_ch, stride, downsample)]
        self.in_channels = out_ch
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


def pretrained_mobilenet_v2(num_classes: int) -> nn.Module:
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.features[-3:].parameters():
        param.requires_grad = True

    model.classifier = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(model.last_channel, num_classes),
    )
    return model


def pretrained_efficientnet_b0(num_classes: int) -> nn.Module:
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.features[-3:].parameters():
        param.requires_grad = True

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(in_features, num_classes),
    )
    return model

In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in tqdm(loader, desc="    train", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in tqdm(loader, desc="    eval ", leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    total = len(all_labels)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return running_loss / total, acc, f1, np.array(all_preds), np.array(all_labels)


def train_model(model, train_loader, val_loader,
                num_epochs, lr, device, model_name="model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    params_to_update = [p for p in model.parameters() if p.requires_grad]
    trainable = sum(p.numel() for p in params_to_update)
    total_params = sum(p.numel() for p in model.parameters())
    optimizer = optim.AdamW(params_to_update, lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_f1 = 0.0
    best_state = None
    no_improve = 0

    print(f"\n{'='*65}")
    print(f"  Обучение: {model_name}")
    print(f"  Параметров: {total_params:,} (обучаемых: {trainable:,})")
    print(f"  Эпох: {num_epochs}, LR: {lr}, Device: {device}")
    print(f"{'='*65}")

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc, val_f1, _, _ = evaluate(
            model, val_loader, criterion, device
        )
        scheduler.step()

        marker = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            marker = f" **BEST**"
            no_improve = 0
        else:
            no_improve += 1

        print(
            f"  Epoch {epoch:02d}/{num_epochs} | "
            f"Train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"Val loss {val_loss:.4f} acc {val_acc:.3f} F1 {val_f1:.4f}{marker}"
        )

        if best_f1 > 0.85:
            break

        if no_improve >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping: F1 не улучшался {EARLY_STOP_PATIENCE} эпох.")
            break

    # Загружаем лучшие веса
    if best_state is not None:
        model.load_state_dict(best_state)
    model = model.to(device)
    print(f"  Лучший Val F1_macro: {best_f1:.4f}\n")
    return model, best_f1

In [9]:
paths, labels, classes, class_to_idx = build_data(IMAGES_DIR)
num_classes = len(classes)
train_loader, val_loader, test_loader = create_dataloaders(paths, labels, num_classes)

  Всего изображений (после субсэмплинга): 50797
  Классы (11): ['Beige', 'Black', 'Blue', 'Brown', 'Green', 'Grey', 'Orange', 'Red', 'Silver', 'White', 'Yellow']

  Распределение по цветам:
  Beige 557
  Black 10000
  Blue 7717
  Brown 836
  Green 709
  Grey 8618
  Orange 518
  Red 5605
  Silver 7119
  White 8497
  Yellow 621

  Train: 32509,  Val: 8128,  Test: 10160


In [10]:
model_custom = CustomResNet(num_classes)
model_custom, _ = train_model(
    model_custom, train_loader, val_loader,
    num_epochs=NUM_EPOCHS_CUSTOM, lr=LR_CUSTOM, device=DEVICE,
    model_name="Custom ResNet",
)


  Обучение: Custom ResNet
  Параметров: 2,797,867 (обучаемых: 2,797,867)
  Эпох: 20, LR: 0.003, Device: cuda


  Epoch 01/20 | Train loss 1.1856 acc 0.564 | Val loss 1.0944 acc 0.607 F1 0.5525 **BEST**


  Epoch 02/20 | Train loss 0.8774 acc 0.686 | Val loss 0.7808 acc 0.721 F1 0.6613 **BEST**


  Epoch 03/20 | Train loss 0.7259 acc 0.745 | Val loss 0.9631 acc 0.655 F1 0.5774


  Epoch 04/20 | Train loss 0.6429 acc 0.773 | Val loss 0.7081 acc 0.750 F1 0.6648 **BEST**


...

  Epoch 16/20 | Train loss 0.1752 acc 0.941 | Val loss 0.4318 acc 0.871 F1 0.8185 **BEST**


  Epoch 17/20 | Train loss 0.1622 acc 0.947 | Val loss 0.4328 acc 0.874 F1 0.8174


  Epoch 18/20 | Train loss 0.1572 acc 0.946 | Val loss 0.4267 acc 0.872 F1 0.8182


  Epoch 19/20 | Train loss 0.1504 acc 0.951 | Val loss 0.4223 acc 0.875 F1 0.8215 **BEST**


  Epoch 20/20 | Train loss 0.1483 acc 0.950 | Val loss 0.4226 acc 0.876 F1 0.8237 **BEST**
  Лучший Val F1_macro: 0.8237



In [11]:
# 2a. MobileNetV2
model_mob = pretrained_mobilenet_v2(num_classes)
model_mob, _ = train_model(
    model_mob, train_loader, val_loader,
    num_epochs=20, lr=LR_PRETRAINED, device=DEVICE,
    model_name="MobileNetV2 (pretrained ImageNet)",
)


  Обучение: MobileNetV2 (pretrained ImageNet)
  Параметров: 2,237,963 (обучаемых: 1,220,171)
  Эпох: 20, LR: 0.001, Device: cuda


  Epoch 01/20 | Train loss 0.8197 acc 0.710 | Val loss 0.8183 acc 0.700 F1 0.6345 **BEST**


  Epoch 02/20 | Train loss 0.5694 acc 0.799 | Val loss 0.7410 acc 0.748 F1 0.6899 **BEST**


  Epoch 03/20 | Train loss 0.4762 acc 0.834 | Val loss 0.7033 acc 0.763 F1 0.6880


  Epoch 04/20 | Train loss 0.4178 acc 0.855 | Val loss 0.6590 acc 0.775 F1 0.7141 **BEST**


...

  Epoch 16/20 | Train loss 0.1712 acc 0.939 | Val loss 0.5858 acc 0.826 F1 0.7712 **BEST**


  Epoch 17/20 | Train loss 0.1686 acc 0.940 | Val loss 0.5754 acc 0.827 F1 0.7733 **BEST**


  Epoch 18/20 | Train loss 0.1555 acc 0.943 | Val loss 0.5873 acc 0.827 F1 0.7723


  Epoch 19/20 | Train loss 0.1599 acc 0.941 | Val loss 0.5805 acc 0.828 F1 0.7758 **BEST**


  Epoch 20/20 | Train loss 0.1561 acc 0.944 | Val loss 0.5864 acc 0.829 F1 0.7744
  Лучший Val F1_macro: 0.7758



In [12]:
# 2b. EfficientNet-B0
model_eff = pretrained_efficientnet_b0(num_classes)
model_eff, _ = train_model(
    model_eff, train_loader, val_loader,
    num_epochs=20, lr=LR_PRETRAINED, device=DEVICE,
    model_name="EfficientNet-B0 (pretrained ImageNet)",
)


  Обучение: EfficientNet-B0 (pretrained ImageNet)
  Параметров: 4,021,639 (обучаемых: 3,169,831)
  Эпох: 20, LR: 0.001, Device: cuda


  Epoch 01/20 | Train loss 0.6917 acc 0.761 | Val loss 0.5988 acc 0.798 F1 0.7327 **BEST**


  Epoch 02/20 | Train loss 0.4008 acc 0.862 | Val loss 0.6195 acc 0.798 F1 0.7437 **BEST**


  Epoch 03/20 | Train loss 0.3361 acc 0.882 | Val loss 0.5855 acc 0.809 F1 0.7479 **BEST**


  Epoch 04/20 | Train loss 0.2905 acc 0.901 | Val loss 0.4980 acc 0.844 F1 0.7872 **BEST**


...

  Epoch 16/20 | Train loss 0.0873 acc 0.968 | Val loss 0.4426 acc 0.883 F1 0.8371 **BEST**


  Epoch 17/20 | Train loss 0.0799 acc 0.972 | Val loss 0.4467 acc 0.884 F1 0.8362


  Epoch 18/20 | Train loss 0.0721 acc 0.974 | Val loss 0.4614 acc 0.883 F1 0.8329


  Epoch 19/20 | Train loss 0.0681 acc 0.976 | Val loss 0.4519 acc 0.887 F1 0.8373 **BEST**


  Epoch 20/20 | Train loss 0.0683 acc 0.976 | Val loss 0.4516 acc 0.886 F1 0.8342
  Лучший Val F1_macro: 0.8373



In [16]:
criterion = nn.CrossEntropyLoss()
final_results = {}

for name, model in [
    ("Custom ResNet-18 (с нуля)", model_custom),
    ("MobileNetV2 (pretrained)", model_mob),
    ("EfficientNet-B0 (pretrained)", model_eff),
]:
    _, test_acc, test_f1, preds, labels = evaluate(
        model, test_loader, criterion, DEVICE
    )
    final_results[name] = {"f1_macro": test_f1, "accuracy": test_acc}
    print(f"\n  --- {name} ---")
    print(f"  Test Accuracy : {test_acc:.4f}")
    print(f"  Test F1_macro : {test_f1:.4f}")
    print(f"  F1 > 0.8      : {'✓ ДА' if test_f1 > 0.8 else '✗ НЕТ'}")
    print()
    print(classification_report(
        labels, preds, target_names=classes, zero_division=0
    ))


  --- Custom ResNet-18 (с нуля) ---
  Test Accuracy : 0.8723
  Test F1_macro : 0.8215
  F1 > 0.8      : ✓ ДА

              precision    recall  f1-score   support

       Beige       0.74      0.59      0.66       111
       Black       0.86      0.91      0.88      2000
        Blue       0.90      0.87      0.89      1543
       Brown       0.59      0.71      0.64       167
       Green       0.73      0.76      0.75       142
        Grey       0.83      0.74      0.78      1724
      Orange       0.71      0.74      0.72       104
         Red       0.97      0.97      0.97      1121
      Silver       0.82      0.87      0.84      1424
       White       0.94      0.95      0.94      1700
      Yellow       0.96      0.94      0.95       124

    accuracy                           0.87     10160
   macro avg       0.82      0.82      0.82     10160
weighted avg       0.87      0.87      0.87     10160




  --- MobileNetV2 (pretrained) ---
  Test Accuracy : 0.8355
  Test F1_macro : 0.7851
  F1 > 0.8      : ✗ НЕТ

              precision    recall  f1-score   support

       Beige       0.72      0.57      0.64       111
       Black       0.83      0.80      0.81      2000
        Blue       0.83      0.82      0.83      1543
       Brown       0.55      0.56      0.55       167
       Green       0.70      0.60      0.65       142
        Grey       0.74      0.74      0.74      1724
      Orange       0.71      0.74      0.72       104
         Red       0.98      0.96      0.97      1121
      Silver       0.81      0.85      0.83      1424
       White       0.92      0.96      0.94      1700
      Yellow       0.96      0.97      0.96       124

    accuracy                           0.84     10160
   macro avg       0.79      0.78      0.79     10160
weighted avg       0.83      0.84      0.83     10160




  --- EfficientNet-B0 (pretrained) ---
  Test Accuracy : 0.8859
  Test F1_macro : 0.8404
  F1 > 0.8      : ✓ ДА

              precision    recall  f1-score   support

       Beige       0.80      0.64      0.71       111
       Black       0.87      0.90      0.88      2000
        Blue       0.91      0.88      0.89      1543
       Brown       0.68      0.63      0.65       167
       Green       0.85      0.75      0.80       142
        Grey       0.81      0.80      0.80      1724
      Orange       0.76      0.71      0.74       104
         Red       0.97      0.97      0.97      1121
      Silver       0.85      0.90      0.88      1424
       White       0.97      0.95      0.96      1700
      Yellow       0.96      0.95      0.96       124

    accuracy                           0.89     10160
   macro avg       0.86      0.83      0.84     10160
weighted avg       0.89      0.89      0.89     10160



In [15]:
print(f"  {'Модель':<38} {'Accuracy':>10} {'F1_macro':>10} {'F1>0.8':>8}")
print("  " + "-" * 66)

sorted_results = sorted(
    final_results.items(), key=lambda x: x[1]["f1_macro"], reverse=True
)
for name, m in sorted_results:
    status = "✓" if m["f1_macro"] > 0.8 else "✗"
    print(f"  {name:<38} {m['accuracy']:>10.4f} {m['f1_macro']:>10.4f} {status:>8}")

best_name = sorted_results[0][0]
best_f1 = sorted_results[0][1]["f1_macro"]

  Модель                                   Accuracy   F1_macro   F1>0.8
  ------------------------------------------------------------------
  EfficientNet-B0 (pretrained)               0.8859     0.8404        ✓
  Custom ResNet-18 (с нуля)                  0.8723     0.8215        ✓
  MobileNetV2 (pretrained)                   0.8355     0.7851        ✗


Самописная ResNet и EfficientNet достигли требуемого качеста, а MobileNet не достигла

Давайте разберем модели по отдельности:
* Самописная ResNet [Параметров: 2,797,867 (обучаемых: 2,797,867)] - параметров впритык хватило, чтобы достичь нужного качества, что видно по логам обучения.
* MobileNetV2 [Параметров: 2,237,963 (обучаемых: 1,220,171) + finetuned] - параметров немного меньше, чем у нашей ResNet, а обучаемых параметров ощутимо меньше. На первых итерациях выдает лучшее качество, чем наша ResNet, но вероятно из-за ограниченного числа обучаемых параметров сходится раньше и не достигает требуемого качества.
* EfficientNet-B0 [Параметров: 4,021,639 (обучаемых: 3,169,831) + finetuned] - параметров ощутимо больше, чем у нашей ResNet, обучаемых параметров немного больше. Сразу на первых итерациях показывает достойный f1-скор. Из-за большого количества обучаемых параметров смогла еще лучше подстроиться под новую задачу и достигла хорошего качества.

Итог:\
Предобученные модельки требуют меньшего числа эпох для сходимости и их можно частично обучать (некоторые параметры заморозить).\
Такой подход более эффективен с точки зрения ресурсов и позволяет получать хорошее качество на малом числе обучающих данных.